# Spark Declarative Pipeline

## 1. Spark Declarative Pipeline Overview

### 1. Introducing Declarative Pipelines: Reliable data pipelines made easy
**Simplified Pipeline Authoring**
* **Write SQL or Python**
* **No orchestration logic**: Lakeflow manages the execution plan for you
* **Zero boilerplate**: Error handling & dependency management built-in

**Intelligent Optimization at Scale**
* **Auto-scaling**: Pipelines scale automatically as data volumes grow
* **Self-healing**: Automatic recovery from failures — no manual steps
* **Lower overhead**: Better reliability with much less operational effort

**Unified Batch and Streaming**
- **Process historical and real-time data seamlessly**
- **Adapts and optimizes** for performance and cost
- **Same code handles both batch and streaming**


### 2. Creating a Spark Declarative Pipeline

**Create an ETL Pipeline — From the Workspace Menu**
* From anywhere in your workspace, click the options (⋮) button in the navigation bar.
* Select Create from the context menu.
* Then choose ETL Pipeline from the submenu.
![image_1794907298312.png](./image_1794907298312.png "image_1794907298312.png")

**Create an ETL Pipeline — From Jobs & Pipelines**
* Go directly to the Jobs & Pipelines section in the main navigation bar.
* Click the Create button in the top-right corner.
* Then select ETL Pipeline — Build and run an ETL pipeline with SQL and Python.
![image_1794907424481.png](./image_1794907424481.png "image_1794907424481.png")

## 2. Build Data Pipelines with Apache Spark Declarative Pipelines 

### 1. Dataset Types 

#### Streaming Table

* Streaming Tables are designed specifically for streaming or incremental data processing.
* Increases efficiency and decreases cost.
* Incremental Processing Modes: Supports batch or streaming for incremental data processing (exactly-once processing of files)
* SQL syntax to create streaming tables: CREATE OR REFRESH STREAMING TABLE
* Must use FROM STREAM read_files() to enable incremental streaming reads with checkpointing.
* This syntax leverages Databricks Auto Loader, which automatically tracks new files and ensures reliable, incremental ingestion.
* No Duplicate Reads: File names are guaranteed to be read only once.

Syntax:
```
CREATE OR REFRESH STREAMING TABLE table_name
AS
SELECT ...
FROM STREAM source_table;

#### Materialized Views

* Materialized Views are used to store the results of SQL queries physically.
* When upstream data changes, the Materialized View is refreshed to reflect the latest data.
* Pipeline integration: Automatically maintained by the pipeline
* Refresh mechanism: CREATE OR REFRESH MATERIALIZED VIEW
* Flexible Pipeline Placement: Can be used anywhere in your pipeline, not just in the gold layer.
* Incremental Refresh Capability: Where applicable, results are incrementally refreshed, avoiding a full rebuild when new data arrives. Supported on **Serverless compute**.
* Cost-Based Optimization: Incremental refresh is driven by a cost-based optimizer to power fast and efficient transformations on Serverless compute.
* They are particularly useful for:
    - Aggregations and analytical transformations.
    - Preparing Gold-layer datasets.
    - Complex SQL transformations.
    - Improving query performance.
    - Creating reusable, precomputed datasets.


Syntax:
```
CREATE OR REFRESH MATERIALIZED VIEW view_name
AS
SELECT ...
FROM source_table;
```

##### Incremental Refresh and Cost-Based Optimization

One important capability of Materialized Views is incremental refresh.

Suppose the upstream Silver table contains 10 million records, and only 1,000 new records arrive.

A full recomputation would potentially require processing the entire dataset again.

However, when incremental refresh is supported and selected, Databricks can process the relevant changes and update the Materialized View without rebuilding everything.

**Important concepts:**
- Incremental refresh is not guaranteed for every query.
- The optimizer determines whether incremental refresh is applicable.
- Databricks uses cost-based optimization to choose an appropriate refresh strategy.
- Incremental refresh capabilities depend on supported query patterns, source types, and compute configuration.
- Serverless compute is relevant to the incremental refresh capabilities described in this course.

#### Temporary Views

* Temporary Views are logical representations of SQL queries that exist only within the pipeline's execution context.
* They do not physically store data and are primarily used to simplify intermediate transformations.
* When should we use Temporary Views?
    - You need intermediate transformation logic.
    - You want to simplify complex SQL queries.
    - A dataset is needed only within the pipeline.
    - You do not need the dataset to be available as a persistent Unity Catalog object.

Syntax:
```
CREATE TEMPORARY VIEW view_name AS

SELECT ...
FROM source_table;
```

#### Views 

* A View is a logical representation of a SQL query.
* Ordinary Views do not physically store the query results. Instead, the underlying query is executed when the View is queried.
* It is a persistent catalog object.
* View Limitations in Declarative Pipelines:
    - The pipeline must be a Unity Catalog pipeline.
    - Ordinary Views cannot be defined as streaming queries.
    - Ordinary Views cannot be used as streaming sources for a pipeline.

Syntax:
```
CREATE VIEW catalog.schema.view_name AS

SELECT ...
FROM source_table;
```

### 2. The Declarative Pipeline Graph

* One of the most powerful features of Databricks Declarative Pipelines is automatica dependency resolution.
* Declarative Pipelines use the dependencies defined in SQL or Python code to construct a Pipeline Graph automatically.
* This graph represents:
    - Datasets within the pipeline.
    - Dependencies between datasets.
    - The flow of data between transformations.
    - The execution order determined by those dependencies.


In [0]:
%sql
-- Below code is part of Declarative Pipeline which can be found here in this location: Spark-Data-Engineering-Scenarios/BikeSalesDeclarativePipeline/BusinessPartnerE2Epipeline.sql

-- Bronze
CREATE OR REFRESH STREAMING TABLE
    bikesales.bronze.business_parter_bronze_dp AS
SELECT *
FROM STREAM read_files(
    '/Volumes/bikesales/sales-md/dbv/BusiessPartner/',
    format => 'csv'
);


-- Silver
CREATE OR REFRESH STREAMING TABLE
    bikesales.silver.business_parter_silver_dp AS
SELECT *
FROM STREAM bikesales.bronze.business_parter_bronze_dp;

-- Gold
CREATE OR REFRESH MATERIALIZED VIEW
    bikesales.gold.business_parter_gold_dp AS
SELECT
    count(*) AS total_orders
FROM bikesales.silver.business_parter_silver_dp;

Below is the Pipeline Monitoring View after executing the Declarative Pipeline. You can see databricks has automatically resolved the dependency between different layer and formed a DAG graph.
![image_1794928973707.png](./image_1794928973707.png "image_1794928973707.png")

### 3. Benefits of the Declarative Pipeline Graph 

- Automatic dependency management: No need to manually define the execution order of dependent datasets.
- Dependency-based execution: Upstream dependencies are processed before downstream datasets.
- Simplified development: Developers focus on defining transformations rather than orchestrating execution.
- Better visualization: The graph provides a visual representation of the complete data flow.
- Efficient pipeline execution: Databricks can determine the appropriate execution plan based on the dependency graph.

## 3. Simplified Pipeline Development and Common Pipeline Settings


### 1. Multi-File Editor

The Multi-File Editor is an integrated development environment provided by Databricks for developing Apache Spark Declarative Pipelines. <br>
Key Characteristics
- Provides a centralized environment for pipeline development.
- Supports multiple SQL and Python files.
- Allows developers to organize pipeline logic into modular files.
- Integrates pipeline configuration and development tools.
- Provides visual representation of pipeline dependencies.
- Supports debugging, data previews, and validation.
Supported Development Formats
|Format|Description|
|---|---|
|.sql files|Write pipeline transformations using SQL|
|.py files|Write pipeline transformations using Python|
|Notebooks|Develop pipeline logic using notebooks|

![image_1794937416794.png](./image_1794937416794.png "image_1794937416794.png")

### 2. Key Features of the Multi-File Editor

#### 1. Pipeline Asset Browser
Key functionalities:
- View pipeline files in one place.
- Navigate between SQL and Python files.
- Organize pipeline code.
- Access different components of the pipeline.

![image_1794938139183.png](./image_1794938139183.png "image_1794938139183.png")

#### 2. Multi-File Code Editor
Key functionalities:
- Open and edit multiple files.
- Develop SQL and Python pipeline logic.
- Organize transformations into separate files.
- Switch between files without leaving the editor.
![image_1794939781058.png](./image_1794939781058.png "image_1794939781058.png")

#### 3. Pipeline-Specific Toolbars

The editor provides toolbars containing shortcuts to commonly used pipeline operations.

These help developers quickly access development and pipeline management actions without navigating through multiple screens.

Examples include:
- Running pipeline operations.
- Accessing pipeline settings.
- Previewing data.
- Validating pipeline code.
- Monitoring execution.

#### 4. Interactive DAG (Directed Acyclic Graph)
A DAG represents the relationship between different datasets and transformations in a pipeline.

Benefits:
- Understand dataset dependencies.
- Visualize the flow of data.
- Identify upstream and downstream relationships.
- Understand the impact of changes.
- Simplify pipeline troubleshooting.

![image_1794939964240.png](./image_1794939964240.png "image_1794939964240.png")

#### 5. Data Previews
- Inspect intermediate transformation results.
- Validate SQL and Python logic.
- Identify incorrect transformations.
- Check data before proceeding with full pipeline execution.

#### 6. Execution Insights and Debugging

These capabilities help answer questions such as:
- Which transformation failed?
- What caused the failure?
- Which dataset is affected?
- What happened during pipeline execution?
- How can the issue be resolved?

#### 7. Dry Run and Validation 
- Identify issues earlier in development.
- Validate pipeline changes.
- Reduce unnecessary full pipeline executions.
- Improve development efficiency.

### 3. Pipeline Settings

These settings control different aspects of pipeline execution, including:
- Compute resources.
- Pipeline code files.
- Data storage locations.
- Configuration parameters.
- Environment dependencies.
- Pipeline metadata and tags.

**Accessing Pipeline Settings**

- Open the Declarative Pipeline in the Multi-File Editor.
- Locate the gear icon in the editor.
- Click the icon to open Pipeline Settings.
- Select the configuration section you want to modify.
- Update the required settings.

**Compute**

Compute settings define the execution environment and resources used by the pipeline.

- Compute resources.
- Execution environment.
- Compute configuration.
- Serverless or other supported compute options.
- Performance and resource utilization.

**Code Assets**

Code Assets define which files and code modules are associated with the pipeline.<br>
They allow developers to manage the source code that constitutes the pipeline.<br>
Key considerations:
- SQL files.
- Python files.
- Notebooks.
- Additional code modules.
- Pipeline source organization.

**Configuration Parameters**

Used to supply values to the pipeline.

Examples:
- Source file paths.
- Environment names.
- Processing flags.
- Business-specific parameters.

![image_1794941515986.png](./image_1794941515986.png "image_1794941515986.png")

**Environment Dependencies**

Used to specify additional software libraries or packages required by the pipeline.


### 4. Default Location for Data Assets

* This setting defines the default location where pipeline data assets are stored.
* It is relevant when determining where the pipeline's datasets should be persisted.

![image_1794941490557.png](./image_1794941490557.png "image_1794941490557.png")

### 5. Additional Pipeline Settings

|Setting|Purpose|
|---|---|
|Pipeline Settings|Configure general pipeline behavior and metadata|
|Code Assets|Manage pipeline source files and code modules|
|Default Location for Data Assets|Define the default data storage location|
|Compute|Configure the execution environment|
|Configuration|Define parameters used by pipeline code|
|Environment Dependencies|Specify external packages and libraries|
|Tags|Categorize and organize pipelines|

![image_1794941661964.png](./image_1794941661964.png "image_1794941661964.png")

## 4. Databricks Declarative Pipelines – Data Quality Expectations

### 1. What Are Expectations?

* Apache Spark Declarative Pipelines (formerly Delta Live Tables) provide a built-in data quality mechanism called Expectations.
* Expectations allow developers to define data quality rules that validate incoming records during ETL processing.
* Why Do We Need Expectations?
    - Maintain data quality throughout the pipeline.
    - Identify invalid or unexpected records.
    - Prevent bad data from reaching downstream datasets.
    - Monitor data quality violations.
    - Enforce business rules during data processing.
    - Improve the reliability of ETL pipelines.

#### Expectation Syntax

```
CONSTRAINT constraint_name
EXPECT (column_condition)
[ON VIOLATION action]
```

|Component|Description|
|---|---|
|CONSTRAINT|Defines a data quality rule|
|constraint_name|Unique name identifying the expectation|
|EXPECT|Specifies the condition that records must satisfy|
|column_condition|SQL Boolean expression evaluated for each row|
|ON VIOLATION|Specifies what happens when a record violates the rule|
|action|Defines whether to warn, drop the row, or fail the flow|

### 2. The Three Expectation Violation Actions

##### WARN

* Log Violations, Keep Rows
* WARN is the default behavior when no violation action is specified.
* When a record violates an expectation:
    - The invalid record is still written to the target dataset.
    - The violation is recorded in pipeline metrics.
    - The pipeline continues processing.
    - Data quality metrics provide information about valid and invalid records.
* When should we use WARN?
    - When monitoring data quality without immediately rejecting records.
    - When identifying data quality issues in a new source system.
    - When violations are acceptable temporarily.
    - When we want to observe data quality trends before enforcing stricter rules.

Example:
```
CONSTRAINT valid_notification
EXPECT (notifications IN ('Y', 'N'));
```

##### DROP

* Archive/Exclude Invalid Rows
* DROP removes records that violate the specified expectation.
* When a record violates the condition:
    - The invalid record is discarded.
    - The invalid record is not written to the target dataset.
    - Other valid records continue processing.
    - The number of dropped records is recorded in pipeline metrics.
* When should we use DROP?
    - When invalid records should not enter downstream datasets.
    - When records violating business rules cannot be processed.
    - When rejecting individual records is preferable to stopping the entire flow.

Example:
```
CONSTRAINT valid_date
EXPECT (order_timestamp > "2021-01-01")
ON VIOLATION DROP ROW
```

##### FAIL

* Stop the Flow/Execution on Violations
* FAIL causes the affected pipeline flow to fail when an expectation is violated.
* When a record violates the condition:
    - The affected flow fails.
    - The flow's update is unsuccessful.
    - Other independent flows in the pipeline are not necessarily affected.
    - Manual intervention may be required to resolve the underlying issue and rerun the flow.
* When should we use FAIL?
    - When a violation represents a critical business issue.
    - When mandatory fields are missing.
    - When data integrity must be strictly enforced.
    - When processing incorrect data could lead to serious downstream consequences.
* Note: FAIL UPDATE is a flow-level failure, not necessarily a failure of the entire pipeline. Other independent flows can continue.

Example:
```
CONSTRAINT valid_id
EXPECT (customer_id IS NOT NULL)
ON VIOLATION FAIL UPDATE
```

### 3. Comparison of the Three Violation Actions
|Feature|WARN|DROP|FAIL|
|---|---|---|---|
|Invalid row retained?|Yes|No|No successful update|
|Violation logged?|Yes|Yes|Yes, through failure information|
|Processing continues?|Yes|Yes|Affected flow fails|
|Other independent flows|Continue|Continue|Can continue|
|Default action?|Yes|No|No|
|SQL action|Default|DROP ROW|FAIL UPDATE|

### 4. Implementing Expectations in a Streaming Table

In [0]:
%sql
....>
-- Silver
CREATE OR REFRESH STREAMING TABLE bikesales.silver.business_parter_silver_dp
(
    CONSTRAINT currency_non_euro EXPECT (CURRENCY NOT IN ('EUR')),
    CONSTRAINT phone_no EXPECT (len(PHONENUMBER) >= 10) ON VIOLATION DROP ROW,
    CONSTRAINT bp_id EXPECT (PARTNERID IS NOT NULL) ON VIOLATION FAIL UPDATE
)
 AS
SELECT *
FROM STREAM bikesales.bronze.business_parter_bronze_dp;
<....

![image_1794998186969.png](./image_1794998186969.png "image_1794998186969.png")

### 5. Monitoring Expectations and Data Quality Metrics

Expectations are not only used to enforce data quality. They also provide visibility into the quality of incoming data.

The pipeline records metrics such as:
- Number of records passing expectations.
- Number of records violating expectations.
- Number of records dropped due to expectations.
- Details of failed updates.

These metrics can be inspected through the pipeline's event log and data quality monitoring interfaces.

## 5. Streaming Joins


* Streaming Joins refer to joining datasets that involve streaming data sources within Databricks Declarative Pipelines.
* There are three major join patterns:
    1. Stream-Snapshot Join – Joining a Streaming Table with a static table.
    2. Streaming Tables via Materialized View – Joining two Streaming Tables through a Materialized View.
    3. Stream-Stream Join – Incrementally joining two live streaming sources.

### 1. Stream-Snapshot Join

* A Stream-Snapshot Join involves joining:
    - One Streaming Table.
    - One static table (snapshot or reference table).
- Only new incoming streaming records need to be processed incrementally.
- The static reference table is used to enrich the incoming data.
- The objective is to avoid repeatedly processing the entire streaming dataset.
- Useful for lookup and reference-data enrichment.

How Does Stream-Snapshot Join Work?

As new records arrive in the Streaming Table:
- Databricks processes the newly arriving streaming records.
- It joins them with the static reference table.
- The matching reference information is added to the incoming records.
- The enriched records are written to the target Streaming Table.

![image_1795008369438.png](./image_1795008369438.png "image_1795008369438.png")

### 2. Joining Streaming Tables with a Materialized View

* This approach involves two datasets that receive new records over time.
* The objective is to join the data from both Streaming Tables and maintain the joined results in a Materialized View.
* Both source datasets are Streaming Tables.
* The join is defined inside a Materialized View.
* New data in either upstream dataset can cause the Materialized View to be refreshed.
* The Materialized View maintains the joined results.
* Incremental refresh may be used where supported and applicable.

* How does it work?
    * New records arrive in the both the Streaming Tables.
    * The Materialized View detects changes in its upstream datasets.
    * The joined results are refreshed to reflect the latest upstream data.
    * Downstream consumers can query the Materialized View.

![image_1795008397975.png](./image_1795008397975.png "image_1795008397975.png")

### 3. Stream-Stream Join

* A Stream-Stream Join involves joining two streaming datasets incrementally.
* Unlike a Stream-Snapshot Join, where one dataset is static, both datasets continuously receive new records.
* The objective is to identify relationships between incoming records from two live streams.
* Spark maintains state containing previously received records so that a new order can match a payment that arrives later, or vice versa.
* However, watermarking and time constraints help determine how long records remain available for matching.

Example:
```
SELECT *
FROM orders_stream o
JOIN payments_stream p
    ON o.order_id = p.order_id
   AND p.payment_timestamp >= o.order_timestamp
   AND p.payment_timestamp <=
       o.order_timestamp + INTERVAL 5 MINUTES;
```

### 4. Comparing the Three Join Types

|Feature|Stream-Snapshot|Streaming via Materialized View|Stream-Stream|
|---|---|---|---|
|Source 1|Streaming|Streaming|Streaming|
|Source 2|Static|Streaming|Streaming|
|Join execution|Incremental|Materialized View refresh|Incremental|
|Historical data|New stream records joined with snapshot|Results reflect upstream datasets|Uses maintained streaming state|
|Output|Streaming Table|Materialized View|Streaming result|
|Complexity|Low|Moderate|High|
|Typical purpose|Data enrichment|Consolidating datasets|Real-time event correlation|

![image_1795019681618.png](./image_1795019681618.png "image_1795019681618.png")

## 6.Scheduling, Notifications, and Monitoring

### 1. Schedule the Pipeline

* Scheduling allows us to configure a pipeline to execute automatically at predefined intervals without requiring manual execution.
* Instead of manually triggering the pipeline every time new data arrives, we can configure a schedule based on business requirements.
* Key Characteristics
    - Supports automatic pipeline execution.
    - Allows execution at predefined intervals.
    - Eliminates the need for manual triggers.
    - Ensures data is refreshed regularly.
    - Helps maintain consistent data availability for downstream consumers.

**Scheduling Options**

|Schedule|Description|Example Use Case|
|---|---|---|
|Every 15 minutes|Executes at 15-minute intervals|Frequent data ingestion|
|Hourly|Executes once every hour|Hourly reporting|
|Daily|Executes once every day|Daily batch processing|
|Continuous|Keeps processing new data as it arrives|Near-real-time data processing|

#### 1. Triggered / Scheduled

Pipeline starts at configured intervals, processes available data, and completes its update.<br>
Example: Every 15 minutes or daily.

![image_1795021068026.png](./image_1795021068026.png "image_1795021068026.png")

#### 2. Continuous

Pipeline remains active and continuously processes incoming data.<br>
Example: Streaming workloads requiring continuous processing.

![image_1795021123225.png](./image_1795021123225.png "image_1795021123225.png")

### 2. Set Up Email Notifications

* Email Notifications allow developers and administrators to receive alerts about pipeline execution activities.
* Helps with proactive monitoring and response.
* The pipeline can be configured to send notifications for the following events:
    * Successful Runs
    * Failed Runs
    * Pipeline Starts

#### Notification Configuration

1. Open the pipeline configuration.
2. Navigate to the notification settings.
3. Specify the email addresses of recipients.
4. Select the events for which notifications should be generated.
5. Save the configuration.

![image_1795021521923.png](./image_1795021521923.png "image_1795021521923.png")

### 3. Monitor with the Event Log

* Event Log provided as  detailed view of runtime information.
* Instead of relying only on email notifications, developers can use the Event Log to investigate what happened during pipeline execution.

What Can We Monitor Through the Event Log?

|Information|Purpose|
|---|---|
|Execution Status|Identify successful, failed, and running updates|
|Row Counts|Understand the volume of records processed|
|Errors|Identify execution failures|
|Warnings|Identify potential issues|
|Data Quality Metrics|Monitor expectation violations|
|Performance Metrics|Analyze pipeline execution and processing performance|

#### Pipelines Event Log

- Audit Logs – Track who did what and when, including actions performed on the pipeline and its datasets.
- Data Quality Checks – Monitor expectation or constraint results, including pass/fail counts and violations.
- Pipeline Progress – Track the status of each pipeline run, row counts, completed stages, and execution state.
- Data Lineage – Understand how data flows and transforms through the pipeline, from source ingestion through Bronze, Silver, and Gold.

#### Querying the Declarative Pipeline Event Log

**Default Behavior**
- By default, the Event Log is stored as a hidden Unity Catalog table.
- It is located in the pipeline's default catalog and schema.

**Publish the Event Log to Metastore (Recommended)** <br>
To make the Event Log easily accessible:
- Open the pipeline's Advanced Settings.
- Specify the target catalog and schema.
- Define the Event Log table name.

This publishes the Event Log as a Unity Catalog table.


**Query the Event Log**<br>
Once published, query it using standard SQL:
```
SELECT *
FROM <catalog>.<schema>.<event_log_table_name>;
```

## 7. Change Data Capture (CDC)
